# Stable semantic-local attention

Purpose: evaluate the existing random-feature far kernel and exact-local kernel reliably when subtraction loses the small far mass. The original notebook and learned parameters remain unchanged. Import only the `ahr-export` definition cell, then call `install()` in the experiment worker.

Inputs: tagged notebook package definitions; bounded synthetic CPU fixtures. Outputs: in-memory diagnostic counters and the CPU validation output below. Expected runtime: less than one minute. No training, GPU work, checkpoint, or long run is launched. Resume: restart the project kernel and Run All; installation is restored after checks.

The normal summary path is retained. Numerically suspect receiver rows use the identical hybrid kernel evaluated with FP64 log weights, in chunks of at most 16 receivers. This fallback uses additional work proportional to affected receivers × source count × feature count; its frequency must be reported. No denominator epsilon, weight clipping, or learned-parameter change is introduced. Active nonfinite projections remain an error.


In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_namespace = {}
for _cell in json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _namespace)
_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
"""Numerically stable evaluation of the unchanged semantic-local hybrid kernel."""
import math
import threading
import torch
from fmm_jepa.semantic_attention import SemanticLocalMixer

backend = "semantic-stable-v1"
_ORIGINAL_MIX_HEADS = SemanticLocalMixer.mix_heads
_COUNTER_LOCK = threading.Lock()
_COUNTERS = {name: 0 for name in
             ("calls", "valid_rows", "fallback_calls", "fallback_rows",
              "unhealthy_rows", "negative_far_rows", "cancellation_rows",
              "wide_range_calls", "tiny_denominator_rows")}
FALLBACK_QUERY_CHUNK = 16


def reset_stats():
    """Reset diagnostics only; no model, random-generator, or optimizer state changes."""
    with _COUNTER_LOCK:
        for name in _COUNTERS:
            _COUNTERS[name] = 0


def stats():
    with _COUNTER_LOCK:
        result = dict(_COUNTERS)
    result["backend"] = backend
    result["fallback_fraction"] = result["fallback_rows"] / max(1, result["valid_rows"])
    return result


def get_stats():
    """JSON-serializable snapshot; alias retained for experiment integrations."""
    return stats()


def _count(**values):
    with _COUNTER_LOCK:
        for name, value in values.items():
            _COUNTERS[name] += int(value)


def _log_space_rows(self, q, k, v, indices, valid, grid, row_coordinates):
    """Evaluate selected receivers directly; no full V-by-V tensor is constructed.

    Each chunk is at most FALLBACK_QUERY_CHUNK receivers. The far log weight
    is logsumexp of the original random-feature products. Exact local weights
    replace those far weights at the original-grid neighborhood. Softmax only
    applies a common row scale and changes neither relative weight nor kernel.
    """
    results = []
    head_dim = q.shape[-1]
    cols = int(grid[1])
    leaf = self.local_leaf_size
    normalizer = 0.5 * math.log(self.semantic_features)
    scale = head_dim ** -0.25
    for offset in range(0, len(row_coordinates), FALLBACK_QUERY_CHUNK):
        coordinates = row_coordinates[offset:offset + FALLBACK_QUERY_CHUNK]
        batch_index, head_index, query_index = coordinates.unbind(-1)
        query = q[batch_index, head_index, query_index].double()
        keys = k[batch_index, head_index].double()
        values = v[batch_index, head_index].double()
        directions = self.omega[head_index].double()
        query_scaled, keys_scaled = query * scale, keys * scale
        query_log = (torch.einsum("nd,nrd->nr", query_scaled, directions)
                     - 0.5 * query_scaled.square().sum(-1, keepdim=True) - normalizer)
        key_log = (torch.einsum("nvd,nrd->nvr", keys_scaled, directions)
                   - 0.5 * keys_scaled.square().sum(-1, keepdim=True) - normalizer)
        if not bool(torch.isfinite(query_log).all() & torch.isfinite(key_log).all()):
            raise FloatingPointError("semantic-stable-v1: feature logits exceed float64 range")
        approximate_log = torch.logsumexp(query_log[:, None, :] + key_log, dim=-1)
        exact_log = torch.einsum("nd,nvd->nv", query, keys) / math.sqrt(head_dim)
        if not bool(torch.isfinite(exact_log).all()):
            raise FloatingPointError("semantic-stable-v1: local logits exceed float64 range")
        source_index = indices[batch_index]
        receiver_index = indices[batch_index, query_index]
        source_row = source_index.div(cols, rounding_mode="floor").div(leaf, rounding_mode="floor")
        source_col = source_index.remainder(cols).div(leaf, rounding_mode="floor")
        receiver_row = receiver_index.div(cols, rounding_mode="floor").div(leaf, rounding_mode="floor")
        receiver_col = receiver_index.remainder(cols).div(leaf, rounding_mode="floor")
        local = ((source_row - receiver_row[:, None]).abs() <= self.local_radius) & (
                 (source_col - receiver_col[:, None]).abs() <= self.local_radius)
        log_weight = torch.where(local, exact_log, approximate_log)
        log_weight = log_weight.masked_fill(~valid[batch_index], -torch.inf)
        weights = torch.softmax(log_weight, dim=-1)
        output = torch.einsum("nv,nvd->nd", weights, values)
        if not bool(torch.isfinite(output).all()):
            raise FloatingPointError("semantic-stable-v1: nonfinite direct log-space readout")
        results.append(output.to(q.dtype))
    return torch.cat(results, dim=0)


def _fallback_all(self, q, k, v, indices, valid, grid, active_rows):
    coordinates = active_rows.nonzero(as_tuple=False)
    _count(fallback_calls=1, fallback_rows=len(coordinates), wide_range_calls=1)
    # Output remains connected to every input, including an all-invalid batch.
    output = q * 0 + k * 0 + v * 0
    if len(coordinates):
        repaired = _log_space_rows(self, q, k, v, indices, valid, grid, coordinates)
        output = output.index_put(tuple(coordinates.unbind(-1)), repaired)
    return output


def stable_mix_heads(self, q, k, v, indices, valid, grid):
    """Original summary/correction path plus a measured direct numerical fallback."""
    batch, heads, count, head_dim = q.shape
    if count == 0:
        return v * 0
    dtype = torch.float64 if q.dtype == torch.float64 else torch.float32
    q, k, v = q.to(dtype), k.to(dtype), v.to(dtype)
    active = valid[:, None, :, None]
    q, k, v = (torch.where(active, tensor, torch.zeros_like(tensor)) for tensor in (q, k, v))
    if not bool(torch.isfinite(q).all() & torch.isfinite(k).all() & torch.isfinite(v).all()):
        raise FloatingPointError("semantic-stable-v1: nonfinite projected values at valid positions")
    active_rows = valid[:, None, :].expand(batch, heads, count)
    _count(calls=1, valid_rows=int(valid.sum()) * heads)
    neighbors, neighbor_valid = self._packed_neighbors(indices, valid, grid)
    neighbor_k = self._gather_sources(k, neighbors)
    neighbor_v = self._gather_sources(v, neighbors)
    raw_local_scores = (q.unsqueeze(-2) * neighbor_k).sum(-1) / math.sqrt(head_dim)
    pair_valid = neighbor_valid[:, None]
    scale = head_dim ** -0.25
    omega = self.omega.to(dtype)
    qs, ks = q * scale, k * scale
    normalizer = 0.5 * math.log(self.semantic_features)
    q_log = torch.einsum("bhvd,hsd->bhvs", qs, omega) - 0.5 * qs.square().sum(-1, keepdim=True) - normalizer
    k_log = torch.einsum("bhvd,hsd->bhvs", ks, omega) - 0.5 * ks.square().sum(-1, keepdim=True) - normalizer
    if not bool(torch.isfinite(q_log).all() & torch.isfinite(k_log).all() &
                torch.isfinite(raw_local_scores).all()):
        return _fallback_all(self, q, k, v, indices, valid, grid, active_rows)
    local_scores = raw_local_scores.masked_fill(~pair_valid, -torch.inf)
    key_shift = k_log.masked_fill(~active, -torch.inf).amax(dim=(-2, -1), keepdim=True)
    key_shift = torch.where(valid.any(-1)[:, None, None, None], key_shift, torch.zeros_like(key_shift)).detach()
    query_shift = torch.maximum(q_log.amax(-1, keepdim=True),
                                local_scores.amax(-1, keepdim=True) - key_shift).detach()
    query_shift = torch.where(active, query_shift, torch.zeros_like(query_shift))
    # Mask before exp: discarded padding must not create an infinity internally.
    q_features = torch.exp((q_log - query_shift).masked_fill(~active, -torch.inf))
    k_features = torch.exp((k_log - key_shift).masked_fill(~active, -torch.inf))
    summary = torch.matmul(k_features.transpose(-2, -1), v)
    if not bool(torch.isfinite(summary).all()):
        return _fallback_all(self, q, k, v, indices, valid, grid, active_rows)
    total_num = torch.matmul(q_features, summary)
    total_den = (q_features * k_features.sum(-2, keepdim=True)).sum(-1)
    local_features = self._gather_sources(k_features, neighbors)
    local_approx = (q_features.unsqueeze(-2) * local_features).sum(-1)
    local_approx = torch.where(pair_valid, local_approx, torch.zeros_like(local_approx))
    local_exact = torch.exp((local_scores - query_shift - key_shift).masked_fill(~pair_valid, -torch.inf))
    far_num = total_num - (local_approx.unsqueeze(-1) * neighbor_v).sum(-2)
    far_den = total_den - local_approx.sum(-1)
    all_local = neighbor_valid.sum(-1).eq(valid.sum(-1, keepdim=True))[:, None]
    far_num = torch.where(all_local[..., None], torch.zeros_like(far_num), far_num)
    far_den = torch.where(all_local, torch.zeros_like(far_den), far_den)
    numerator = far_num + (local_exact.unsqueeze(-1) * neighbor_v).sum(-2)
    denominator = far_den + local_exact.sum(-1)
    healthy = torch.isfinite(numerator).all(-1) & torch.isfinite(denominator) & (denominator > 0)
    tolerance = 64 * torch.finfo(dtype).eps * total_den.detach().abs()
    negative_far = (far_den < 0) & ~all_local & active_rows
    cancellation = (denominator <= tolerance) & ~all_local & active_rows
    unhealthy = ~healthy & active_rows
    # A positive subnormal divisor can have a finite quotient but overflowing
    # backward intermediates. This guard also applies when every source is local.
    tiny_denominator = (denominator < math.sqrt(torch.finfo(dtype).tiny)) & active_rows
    suspect = unhealthy | negative_far | cancellation | tiny_denominator
    # Never create 0/0 or inf/0 and then mask its output: such gradients can leak.
    regular = active_rows & ~suspect
    safe_numerator = torch.where(regular[..., None], numerator, torch.zeros_like(numerator))
    safe_denominator = torch.where(regular, denominator, torch.ones_like(denominator))
    output = safe_numerator / safe_denominator.unsqueeze(-1)
    if bool(suspect.any()):
        coordinates = suspect.nonzero(as_tuple=False)
        _count(fallback_calls=1, fallback_rows=len(coordinates),
               unhealthy_rows=int(unhealthy.sum()), negative_far_rows=int(negative_far.sum()),
               cancellation_rows=int(cancellation.sum()),
               tiny_denominator_rows=int(tiny_denominator.sum()))
        repaired = _log_space_rows(self, q, k, v, indices, valid, grid, coordinates)
        output = output.index_put(tuple(coordinates.unbind(-1)), repaired)
    if not bool(torch.isfinite(output).all()):
        raise FloatingPointError("semantic-stable-v1: nonfinite output after numerical repair")
    return torch.where(active, output, torch.zeros_like(output))


def install():
    """Install for existing/new mixer instances; parameters and state_dict are untouched."""
    current = SemanticLocalMixer.mix_heads
    if current is stable_mix_heads:
        return backend
    if current is not _ORIGINAL_MIX_HEADS:
        raise RuntimeError("Refusing to replace an unrelated semantic attention implementation")
    SemanticLocalMixer.mix_heads = stable_mix_heads
    return backend


def restore():
    """Restore only our own replacement; never overwrite an unrelated installer."""
    current = SemanticLocalMixer.mix_heads
    if current is stable_mix_heads:
        SemanticLocalMixer.mix_heads = _ORIGINAL_MIX_HEADS
    elif current is not _ORIGINAL_MIX_HEADS:
        raise RuntimeError("Cannot restore while an unrelated semantic implementation is installed")
    return "original-semantic"


## Bounded CPU checks

The first fixture forces independent feature shifts to underflow despite a uniform, well-defined answer. The second compares forward and backward values with an independent small FP64 hybrid oracle. These checks do not establish GPU throughput or end-to-end training stability; the captured failing batch and short checkpoint-resume tests are validated separately.


In [ ]:
def _oracle(module, q, k, v, indices, valid, grid):
    """Independent small dense FP64 hybrid weights, used only for CPU validation."""
    q, k, v = q.double(), k.double(), v.double()
    d = q.shape[-1]
    qs, ks = q * d ** -0.25, k * d ** -0.25
    omega = module.omega.double()
    q_log = torch.einsum("bhid,hrd->bhir", qs, omega) - qs.square().sum(-1, keepdim=True) / 2 - math.log(module.semantic_features) / 2
    k_log = torch.einsum("bhjd,hrd->bhjr", ks, omega) - ks.square().sum(-1, keepdim=True) / 2 - math.log(module.semantic_features) / 2
    approximate = torch.logsumexp(q_log.unsqueeze(-2) + k_log.unsqueeze(-3), dim=-1)
    exact = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d)
    rows = indices.div(grid[1], rounding_mode="floor").div(module.local_leaf_size, rounding_mode="floor")
    cols = indices.remainder(grid[1]).div(module.local_leaf_size, rounding_mode="floor")
    near = ((rows[:, :, None] - rows[:, None, :]).abs() <= module.local_radius) & ((cols[:, :, None] - cols[:, None, :]).abs() <= module.local_radius)
    logits = torch.where(near[:, None], exact, approximate).masked_fill(~valid[:, None, None], -torch.inf)
    return torch.matmul(torch.softmax(logits, -1), v)


def run_cpu_checks():
    old_threads = torch.get_num_threads()
    saved_rng = torch.get_rng_state()
    torch.set_num_threads(2)
    torch.manual_seed(9211)
    results = {}
    try:
        module = SemanticLocalMixer(1, 1, 2, 2, 1, 1)
        module.omega.copy_(torch.tensor([[[4.0], [-4.0]]]))
        q = torch.full((1, 1, 2, 1), 16.0, requires_grad=True)
        k = torch.full((1, 1, 2, 1), -16.0, requires_grad=True)
        v = torch.tensor([[[[2.0], [6.0]]]], requires_grad=True)
        indices = torch.tensor([[0, 1]]); valid = torch.ones_like(indices, dtype=torch.bool)
        try:
            _ORIGINAL_MIX_HEADS(module, q, k, v, indices, valid, (1, 2))
        except FloatingPointError:
            original_failed = True
        else:
            original_failed = False
        assert original_failed, "Analytical underflow fixture must reproduce the original failure"
        reset_stats()
        output = stable_mix_heads(module, q, k, v, indices, valid, (1, 2))
        reference = _oracle(module, q, k, v, indices, valid, (1, 2))
        torch.testing.assert_close(output.double(), reference, rtol=0, atol=1e-12)
        gradients = torch.autograd.grad(output.sum(), (q, k, v), retain_graph=True)
        reference_gradients = torch.autograd.grad(reference.sum(), (q, k, v))
        for actual, expected in zip(gradients, reference_gradients):
            assert bool(torch.isfinite(actual).all())
            torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)
        results["analytical_underflow"] = {"status": "PASS", "output": output.detach().flatten().tolist(), "stats": stats()}

        tiny_results = []
        for token_count in (1, 8):
            module = SemanticLocalMixer(1, 1, 2, 4, 1, 1)
            module.omega.copy_(torch.tensor([[[4.0], [-4.0]]]))
            q = torch.full((1, 1, token_count, 1), 13.0, requires_grad=True)
            k = torch.full((1, 1, token_count, 1), -13.0, requires_grad=True)
            v = torch.full((1, 1, token_count, 1), 0.3, requires_grad=True)
            indices = torch.arange(token_count)[None]
            valid = torch.ones_like(indices, dtype=torch.bool)
            reset_stats()
            output = stable_mix_heads(module, q, k, v, indices, valid, (1, token_count))
            reference = _oracle(module, q, k, v, indices, valid, (1, token_count))
            torch.testing.assert_close(output.double(), reference, rtol=2e-7, atol=1e-8)
            gradients = torch.autograd.grad(output.sum(), (q, k, v), retain_graph=True)
            reference_gradients = torch.autograd.grad(reference.sum(), (q, k, v))
            for actual, expected in zip(gradients, reference_gradients):
                assert bool(torch.isfinite(actual).all())
                torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)
            assert stats()["tiny_denominator_rows"] == token_count
            tiny_results.append({"tokens": token_count, "status": "PASS", "stats": stats()})
        results["tiny_positive_denominator_backward"] = tiny_results

        module = SemanticLocalMixer(8, 2, 8, 1, 0, 19).double()
        q, k, v = [torch.randn(1, 2, 5, 4, dtype=torch.float64, requires_grad=True) for _ in range(3)]
        indices = torch.tensor([[0, 3, 5, 8, 11]]); valid = torch.ones_like(indices, dtype=torch.bool)
        output = stable_mix_heads(module, q, k, v, indices, valid, (3, 4))
        reference = _oracle(module, q, k, v, indices, valid, (3, 4))
        torch.testing.assert_close(output, reference, rtol=2e-10, atol=2e-11)
        cotangent = torch.randn_like(output)
        gradients = torch.autograd.grad((output * cotangent).sum(), (q, k, v), retain_graph=True)
        reference_gradients = torch.autograd.grad((reference * cotangent).sum(), (q, k, v))
        for actual, expected in zip(gradients, reference_gradients):
            torch.testing.assert_close(actual, expected, rtol=2e-9, atol=2e-10)
        results["moderate_forward_gradients"] = {"status": "PASS", "maximum_absolute_error": float((output-reference).detach().abs().max())}

        before = {name: value.clone() for name, value in module.state_dict().items()}
        assert install() == backend and install() == backend
        assert SemanticLocalMixer.mix_heads is stable_mix_heads
        restore()
        assert SemanticLocalMixer.mix_heads is _ORIGINAL_MIX_HEADS
        for name, value in module.state_dict().items():
            assert torch.equal(before[name], value)
        results["install_restore_state"] = {"status": "PASS"}
        assert not torch.cuda.is_initialized()
        return results
    finally:
        restore()
        torch.set_rng_state(saved_rng)
        torch.set_num_threads(old_threads)

CPU_VALIDATION = run_cpu_checks()
print(json.dumps(CPU_VALIDATION, indent=2))
